In [ ]:
# ==========================================
# PHASE 06 - STEP 01
# SETUP & HEIGHT TRACE EXTRACTION
# (DEPLOYMENT-ALIGNED: YOLO RE-RUN)
# ==========================================

# -------------------------------
# 0. INSTALL (ONLY IF NEEDED)
# -------------------------------
!pip install -q ultralytics

# -------------------------------
# 1. IMPORTS
# -------------------------------
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import drive

# -------------------------------
# 2. MOUNT GOOGLE DRIVE
# -------------------------------
drive.mount('/content/drive')

# -------------------------------
# 3. PATH CONFIGURATION (LOCKED)
# -------------------------------
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"

# Demo on one fall video (as in old Phase 6)
VIDEO_DIR = os.path.join(BASE_PATH, "GUB-STFN-Fall-Dataset", "Slip")

# Old Phase-6 output directory (KEEP SAME)
PHASE6_DIR = os.path.join(BASE_PATH, "Results", "Phase06_Alarm")
os.makedirs(PHASE6_DIR, exist_ok=True)

TRACE_SAVE_PATH = os.path.join(PHASE6_DIR, "height_trace_pixels.npy")
FIG_SAVE_PATH   = os.path.join(PHASE6_DIR, "phase06_height_trace.png")

# -------------------------------
# 4. LOAD YOLOv8-POSE
# -------------------------------
print("Loading YOLOv8-Pose...")
pose_model = YOLO("yolov8n-pose.pt")
print("✅ YOLOv8-Pose loaded")

# -------------------------------
# 5. SELECT TARGET VIDEO
# -------------------------------
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]
if not video_files:
    raise FileNotFoundError("No .mp4 video found in Slip folder.")

video_name = video_files[0]  # auto-pick first (same assumption as old Phase 6)
video_path = os.path.join(VIDEO_DIR, video_name)
print(f"Processing video: {video_name}")

# -------------------------------
# 6. EXTRACT HEIGHT TRACE (HEAD–HIP)
# -------------------------------
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0 or np.isnan(fps):
    fps = 30.0  # safe fallback

height_trace = []
print("Extracting height trace...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, verbose=False)
    h_val = np.nan  # default if no pose

    if results and results[0].keypoints is not None:
        kp = results[0].keypoints.xy.cpu().numpy()
        conf = results[0].keypoints.conf.cpu().numpy()

        if len(kp) > 0:
            # choose person with highest mean confidence
            idx = np.argmax(np.mean(conf, axis=1))

            # Head (0), Hip center (11,12)
            head_y = kp[idx][0, 1]
            hip_y  = (kp[idx][11, 1] + kp[idx][12, 1]) / 2.0
            h_val = hip_y - head_y

    height_trace.append(h_val)

cap.release()

height_trace = np.array(height_trace)
print(f"Frames processed: {len(height_trace)}")

# -------------------------------
# 7. SAVE RAW TRACE (MANDATORY)
# -------------------------------
np.save(TRACE_SAVE_PATH, height_trace)
print(f"Saved: {TRACE_SAVE_PATH}")

# -------------------------------
# 8. VISUALIZATION (MANDATORY)
# -------------------------------
plt.figure(figsize=(12, 5))
plt.plot(height_trace, linewidth=2, label="Head–Hip Vertical Distance (px)")
plt.xlabel("Frame Index")
plt.ylabel("Vertical Distance (pixels)")
plt.title(f"Phase 06 – Step 01: Height Trace\nVideo: {video_name}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_SAVE_PATH, dpi=300)
plt.show()

print(f"Saved figure: {FIG_SAVE_PATH}")

# -------------------------------
# 9. DONE
# -------------------------------
print("\n✅ PHASE 06 – STEP 01 COMPLETE")
print("Outputs:")
print(" - height_trace_pixels.npy")
print(" - phase06_height_trace.png")
print("Ready for Phase 06 – Step 02")

In [ ]:
# ==========================================
# PHASE 06 - STEP 02
# FALL-DOWN FRAME + ROBUST POST-FALL ALARM LOGIC
# (ANTI-GLITCH STATE-BASED VERIFICATION)
# ==========================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1. PATHS (LOCKED)
# -------------------------------
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
PHASE6_DIR = os.path.join(BASE_PATH, "Results", "Phase06_Alarm")
VIDEO_DIR  = os.path.join(BASE_PATH, "GUB-STFN-Fall-Dataset", "Slip")

TRACE_PATH = os.path.join(PHASE6_DIR, "height_trace_pixels.npy")
FIG_SAVE_PATH = os.path.join(PHASE6_DIR, "phase06_alarm_decision.png")
LOG_SAVE_PATH = os.path.join(PHASE6_DIR, "phase06_alarm_log.txt")

# -------------------------------
# 2. LOAD HEIGHT TRACE
# -------------------------------
if not os.path.exists(TRACE_PATH):
    raise FileNotFoundError("Run Phase 06 – Step 01 first.")

height_trace = np.load(TRACE_PATH)
total_frames = len(height_trace)

# -------------------------------
# 3. READ FPS FROM VIDEO
# -------------------------------
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]
video_path = os.path.join(VIDEO_DIR, video_files[0])

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if fps <= 0 or np.isnan(fps):
    fps = 30.0

# -------------------------------
# 4. STANDING HEIGHT ESTIMATION
# -------------------------------
valid_heights = height_trace[~np.isnan(height_trace)]
standing_height = np.mean(valid_heights[:30])  # roadmap-aligned
lay_threshold = 0.4 * standing_height

# -------------------------------
# 5. FALL-DOWN FRAME DETECTION
# -------------------------------
fall_down_frame = None
for i, h in enumerate(height_trace):
    if not np.isnan(h) and h < lay_threshold:
        fall_down_frame = i
        break

if fall_down_frame is None:
    final_decision = "NO ALARM (No confirmed lay-down)"

else:
    # -------------------------------
    # 6. ROBUST POST-FALL VERIFICATION
    # -------------------------------
    REQUIRED_FRAMES = int(5 * fps)     # 5 seconds
    RECOVERY_FRAMES = 15               # anti-glitch buffer

    down_counter = 0
    recovery_buffer = 0
    alarm_triggered = False

    for j in range(fall_down_frame, total_frames):
        h = height_trace[j]

        if not np.isnan(h) and h < lay_threshold:
            down_counter += 1
            recovery_buffer = 0

            if down_counter >= REQUIRED_FRAMES:
                alarm_triggered = True
                break

        else:
            recovery_buffer += 1

            if recovery_buffer > RECOVERY_FRAMES:
                final_decision = "NO ALARM (Recovered)"
                break

    else:
        # End of video reached
        if recovery_buffer > 0:
            final_decision = "NO ALARM (Benefit of Doubt)"
        elif down_counter < REQUIRED_FRAMES:
            final_decision = "NO ALARM (Insufficient Duration)"
        else:
            final_decision = "ALARM"

    if alarm_triggered:
        final_decision = "ALARM"

# -------------------------------
# 7. SAVE LOG (THESIS EVIDENCE)
# -------------------------------
with open(LOG_SAVE_PATH, "w") as f:
    f.write("PHASE 06 – ROBUST POST-FALL ALARM LOG\n")
    f.write("-----------------------------------\n")
    f.write(f"Video: {video_files[0]}\n")
    f.write(f"FPS: {fps:.2f}\n")
    f.write(f"Standing Height (px): {standing_height:.2f}\n")
    f.write(f"Lay Threshold (40%): {lay_threshold:.2f}\n")
    f.write(f"Fall-Down Frame: {fall_down_frame}\n")
    f.write(f"Final Decision: {final_decision}\n")

print("Alarm log saved:", LOG_SAVE_PATH)

# -------------------------------
# 8. VISUALIZATION
# -------------------------------
plt.figure(figsize=(12, 5))
plt.plot(height_trace, linewidth=2, label="Height Trace")
plt.axhline(standing_height, color="green", linestyle="--", label="Standing Height")
plt.axhline(lay_threshold, color="red", linestyle="--", label="Lay Threshold")

if fall_down_frame is not None:
    plt.axvline(fall_down_frame, color="black", linestyle=":", label="Fall-Down Frame")

plt.title(f"Phase 06 – Robust Alarm Decision\n{final_decision}")
plt.xlabel("Frame Index")
plt.ylabel("Vertical Distance (pixels)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_SAVE_PATH, dpi=300)
plt.show()

print("Decision figure saved:", FIG_SAVE_PATH)

print("\n✅ PHASE 06 – STEP 02 COMPLETE")
print("Final Decision:", final_decision)

In [ ]:
# ==========================================
# PHASE 06 - STEP 03
# THESIS SUMMARY ARTIFACTS (CSV + TXT)
# ==========================================

import os
import cv2
import numpy as np
import pandas as pd

# -------------------------------
# 1. PATHS (LOCKED)
# -------------------------------
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
PHASE6_DIR = os.path.join(BASE_PATH, "Results", "Phase06_Alarm")
VIDEO_DIR  = os.path.join(BASE_PATH, "GUB-STFN-Fall-Dataset", "Slip")

TRACE_PATH = os.path.join(PHASE6_DIR, "height_trace_pixels.npy")
LOG_PATH   = os.path.join(PHASE6_DIR, "phase06_alarm_log.txt")

CSV_PATH   = os.path.join(PHASE6_DIR, "phase06_summary.csv")
TXT_PATH   = os.path.join(PHASE6_DIR, "phase06_summary.txt")

# -------------------------------
# 2. LOAD REQUIRED FILES
# -------------------------------
if not os.path.exists(TRACE_PATH) or not os.path.exists(LOG_PATH):
    raise FileNotFoundError("Run Phase 06 – Step 01 and Step 02 first.")

height_trace = np.load(TRACE_PATH)

# Read the log file from Step 2
with open(LOG_PATH, "r") as f:
    log_lines = f.read().splitlines()

# Function to safely read value from log
def read_value(prefix):
    for line in log_lines:
        if line.startswith(prefix):
            return line.split(":", 1)[1].strip()
    return None

# Safely read necessary values
standing_height = float(read_value("Standing Height (px)"))
lay_threshold   = float(read_value("Lay Threshold (40%)"))
fall_down_frame = read_value("Fall-Down Frame")
final_decision  = read_value("Final Decision")

# Check if fall_down_frame is correctly set as None or integer
if fall_down_frame == "None":
    fall_down_frame = None
else:
    fall_down_frame = int(fall_down_frame)

# -------------------------------
# 3. VIDEO METADATA
# -------------------------------
# Auto-detect the same video used in Steps 1 & 2
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]
video_name = video_files[0]

cap = cv2.VideoCapture(os.path.join(VIDEO_DIR, video_name))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if fps <= 0 or np.isnan(fps):
    fps = 30.0

# Calculate video duration (seconds)
duration_sec = len(height_trace) / fps

# -------------------------------
# 4. BUILD CSV SUMMARY (REPORT)
# -------------------------------
summary_df = pd.DataFrame([{
    "Video_Name": video_name,
    "Total_Frames": len(height_trace),
    "Duration_sec": round(duration_sec, 2),
    "FPS": round(fps, 2),
    "Standing_Height_px": round(standing_height, 2),
    "Lay_Threshold_px": round(lay_threshold, 2),
    "Fall_Down_Frame": fall_down_frame,
    "Final_Alarm_Decision": final_decision
}])

# Save the CSV summary
summary_df.to_csv(CSV_PATH, index=False)
display(summary_df)
print(f"CSV summary saved: {CSV_PATH}")

# -------------------------------
# 5. WRITE TEXT SUMMARY (REPORT)
# -------------------------------
with open(TXT_PATH, "w") as f:
    f.write("PHASE 06 – POST-FALL ALARM SUMMARY\n")
    f.write("----------------------------------\n")
    f.write(f"Video Name           : {video_name}\n")
    f.write(f"Total Frames         : {len(height_trace)}\n")
    f.write(f"Duration (sec)       : {duration_sec:.2f}\n")
    f.write(f"FPS                  : {fps:.2f}\n")
    f.write(f"Standing Height (px) : {standing_height:.2f}\n")
    f.write(f"Lay Threshold (px)   : {lay_threshold:.2f}\n")
    f.write(f"Fall-Down Frame      : {fall_down_frame}\n")
    f.write(f"Final Decision       : {final_decision}\n")

print(f"Text summary saved: {TXT_PATH}")

print("\n✅ PHASE 06 – STEP 03 COMPLETE")
print("Thesis summary artifacts generated.")

In [ ]:
# ==========================================
# PHASE 06 - STEP 04
# FINAL PRESENTATION FIGURE (FIXED & SAFE)
# ==========================================

import os
import numpy as np
import matplotlib.pyplot as plt

# 1. PATHS
BASE_PATH = "/content/drive/MyDrive/Human-Fall-Recognition"
PHASE6_DIR = os.path.join(BASE_PATH, "Results", "Phase06_Alarm")
TRACE_PATH = os.path.join(PHASE6_DIR, "height_trace_pixels.npy")
LOG_PATH   = os.path.join(PHASE6_DIR, "phase06_alarm_log.txt")
FINAL_FIG_PATH = os.path.join(PHASE6_DIR, "phase06_presentation.png")

# 2. LOAD DATA
if not os.path.exists(TRACE_PATH) or not os.path.exists(LOG_PATH):
    raise FileNotFoundError("Run Step 01–03 before Step 04.")

height_trace = np.load(TRACE_PATH)

with open(LOG_PATH, "r") as f:
    lines = f.read().splitlines()

# Function to safely read value from log
def get_val(key):
    for l in lines:
        if l.startswith(key):
            return l.split(":", 1)[1].strip()
    return None

# Safely read necessary values
standing_height = float(get_val("Standing Height (px)"))
lay_threshold   = float(get_val("Lay Threshold (40%)"))
fall_down_frame = get_val("Fall-Down Frame")
final_decision  = get_val("Final Decision")

# Check if fall_down_frame is correctly set as None or integer
if fall_down_frame == "None":
    fall_down_frame = None
else:
    fall_down_frame = int(fall_down_frame)

# 3. PLOT
plt.figure(figsize=(14, 6))

# Main Trace
plt.plot(height_trace, color="black", linewidth=2, label="Head–Hip Vertical Distance")

# Thresholds
plt.axhline(standing_height, color="green", linestyle="--", label="Standing Height")
plt.axhline(lay_threshold, color="red", linestyle="--", label="Lay Threshold")

# --- CRITICAL FIX: Only draw logic if a fall exists ---
if fall_down_frame is not None:
    # 1. Draw Blue Line
    plt.axvline(fall_down_frame, color="blue", linestyle=":", linewidth=2, label="Fall-Down Frame")

    # 2. Determine Color based on Decision
    if final_decision == "ALARM":
        status_color = "red"
        status_label = "ALARM TRIGGERED"
    elif final_decision == "NO ALARM (Recovered)":
        status_color = "green"
        status_label = "RECOVERED"
    else:
        status_color = "yellow"  # Optional: Remove if you don’t want orange
        status_label = "MONITORING (No Alarm)"

    # 3. Draw Shaded Area (Safe from Crashing)
    plt.axvspan(
        fall_down_frame,
        min(len(height_trace), fall_down_frame + 150),
        color=status_color,
        alpha=0.3,
        label=status_label
    )

plt.title(f"Phase 06 – Alarm Decision\nFinal Decision: {final_decision}", fontsize=14)
plt.xlabel("Frame Index")
plt.ylabel("Vertical Distance (pixels)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FINAL_FIG_PATH, dpi=300)
plt.show()

print("Decision figure saved:", FINAL_FIG_PATH)

# -------------------------------
# 4. DONE
# -------------------------------
print("\n✅ PHASE 06 COMPLETE (LOCKED).")